create multi agent reinforcement learning custom environment that adjust the electricity load and temperature based on the number of people in the room using Openai gym

openai gym multi agent reinforcement learning custom environment that adjust the electricity load and temperature based on the number of people in the room

Create a multi-agent reinforcement learning (MARL) environment using OpenAI Gym for adjusting electricity load and temperature based on the number of people in a room involves several steps. Below, I'll guide you through setting up a custom environment that simulates this scenario.

Step 1: Install Required Libraries
First, make sure you have the necessary libraries installed. You can install them using pip if you haven't already:

In [ ]:
!pip install gym numpy

Step 2: Define the Environment
We'll create a custom environment by subclassing gym.Env. This environment will manage the state, actions, and rewards.

In [ ]:
import gym
from gym import spaces
import numpy as np

class MultiAgentRoomEnv(gym.Env):
    metadata = {'render.modes': ['human']}

    def __init__(self, num_agents=3, max_people=10, max_temp=35, min_temp=15, max_load=1000):
        super(MultiAgentRoomEnv, self).__init__()

        self.num_agents = num_agents
        self.max_people = max_people
        self.max_temp = max_temp
        self.min_temp = min_temp
        self.max_load = max_load

        # Observation space: [number of people, current temperature, current load]
        self.observation_space = spaces.Box(low=np.array([0, self.min_temp, 0]),
                                            high=np.array([self.max_people, self.max_temp, self.max_load]),
                                            dtype=np.float32)

        # Action space: [adjust load, adjust temperature]
        self.action_space = spaces.Box(low=np.array([-1, -1]),
                                       high=np.array([1, 1]),
                                       dtype=np.float32)

        # Initialize state
        self.num_people = 0
        self.current_temp = (self.max_temp + self.min_temp) / 2
        self.current_load = 0

    def step(self, actions):
        """
        Actions: List of actions for each agent, where each action is a tuple (load_adjustment, temp_adjustment)
        """
        rewards = []
        for action in actions:
            load_adjustment, temp_adjustment = action
            self.current_load += load_adjustment
            self.current_temp += temp_adjustment

            # Ensure load and temperature stay within bounds
            self.current_load = np.clip(self.current_load, 0, self.max_load)
            self.current_temp = np.clip(self.current_temp, self.min_temp, self.max_temp)

            # Reward function: penalize for deviations from optimal load and temperature
            reward = -abs(self.current_load - self.num_people * 100) - abs(self.current_temp - 22)
            rewards.append(reward)

        # Update number of people (for simplicity, assume a random change)
        self.num_people = np.random.randint(0, self.max_people + 1)

        # Observation, reward, done, info
        observation = np.array([self.num_people, self.current_temp, self.current_load])
        done = False
        info = {}

        return observation, rewards, done, info

    def reset(self):
        self.num_people = 0
        self.current_temp = (self.max_temp + self.min_temp) / 2
        self.current_load = 0
        return np.array([self.num_people, self.current_temp, self.current_load])

    def render(self, mode='human'):
        print(f"Number of people: {self.num_people}, Current temperature: {self.current_temp}, Current load: {self.current_load}")

    def close(self):
        pass

# Example usage
if __name__ == "__main__":
    env = MultiAgentRoomEnv()
    obs = env.reset()
    print(f"Initial observation: {obs}")

    for _ in range(10):
        actions = [np.random.uniform(-1, 1, 2) for _ in range(env.num_agents)]
        obs, rewards, done, info = env.step(actions)
        print(f"Observation: {obs}, Rewards: {rewards}, Done: {done}, Info: {info}")
        env.render()

Initial observation: [ 0. 25.  0.]
Observation: [ 1.         24.80916797  2.13250907], Rewards: [-3.133575110624691, -4.379777354913861, -4.941677037347928], Done: False, Info: {}
Number of people: 1, Current temperature: 24.809167965330904, Current load: 2.1325090720170246
Observation: [ 1.         25.38135177  0.84957131], Rewards: [-100.6687471589866, -101.73071498184177, -102.53178046008712], Done: False, Info: {}
Number of people: 1, Current temperature: 25.38135176883543, Current load: 0.8495713087483052
Observation: [ 0.         24.46025406  0.        ], Rewards: [-102.43835736106757, -102.59190942769756, -102.46025406318154], Done: False, Info: {}
Number of people: 0, Current temperature: 24.460254063181537, Current load: 0.0
Observation: [ 7.         25.63555812  0.91081931], Rewards: [-3.713654089703791, -3.7120852601308196, -4.54637742885801], Done: False, Info: {}
Number of people: 7, Current temperature: 25.63555811967918, Current load: 0.9108193091788284
Observation: [ 9.

In [ ]:
import gym
from gym import spaces
import numpy as np

class MultiAgentRoomEnv(gym.Env):
    metadata = {'render.modes': ['human']}

    def __init__(self, num_agents=3, max_people=10, max_temp=35, min_temp=15, max_load=1000):
        super(MultiAgentRoomEnv, self).__init__()

        self.num_agents = num_agents
        self.max_people = max_people
        self.max_temp = max_temp
        self.min_temp = min_temp
        self.max_load = max_load

        # Observation space: [number of people, current temperature, current load]
        self.observation_space = spaces.Box(low=np.array([0, self.min_temp, 0]),
                                            high=np.array([self.max_people, self.max_temp, self.max_load]),
                                            dtype=np.float32)

        # Action space: [adjust load, adjust temperature]
        self.action_space = spaces.Box(low=np.array([-1, -1]),
                                       high=np.array([1, 1]),
                                       dtype=np.float32)

        # Initialize state
        self.num_people = 0
        self.current_temp = (self.max_temp + self.min_temp) / 2
        self.current_load = 0
        self.hours = 0

    def step(self, actions):
        """
        Actions: List of actions for each agent, where each action is a tuple (load_adjustment, temp_adjustment)
        """
        rewards = []
        for action in actions:
            load_adjustment, temp_adjustment = action
            self.current_load += load_adjustment
            self.current_temp += temp_adjustment

            # Ensure load and temperature stay within bounds
            self.current_load = np.clip(self.current_load, 0, self.max_load)
            self.current_temp = np.clip(self.current_temp, self.min_temp, self.max_temp)

            # Calculate temperature based on load and hours
            self.current_temp = self.current_load * self.hours

            # Reward function: penalize for deviations from optimal load and temperature
            reward = -abs(self.current_load - self.num_people * 100) - abs(self.current_temp - 22)
            rewards.append(reward)

        # Update number of people (for simplicity, assume a random change)
        self.num_people = np.random.randint(0, self.max_people + 1)

        # Increment hours
        self.hours += 1

        # Observation, reward, done, info
        observation = np.array([self.num_people, self.current_temp, self.current_load])
        done = False
        info = {}

        return observation, rewards, done, info

    def reset(self):
        self.num_people = 0
        self.current_temp = (self.max_temp + self.min_temp) / 2
        self.current_load = 0
        self.hours = 0
        return np.array([self.num_people, self.current_temp, self.current_load])

    def render(self, mode='human'):
        print(f"Number of people: {self.num_people}, Current temperature: {self.current_temp}, Current load: {self.current_load}, Hours: {self.hours}")

    def close(self):
        pass

# Example usage
if __name__ == "__main__":
    env = MultiAgentRoomEnv()
    obs = env.reset()
    print(f"Initial observation: {obs}")

    for _ in range(10):
        actions = [np.random.uniform(-1, 1, 2) for _ in range(env.num_agents)]
        obs, rewards, done, info = env.step(actions)
        print(f"Observation: {obs}, Rewards: {rewards}, Done: {done}, Info: {info}")
        env.render()

Initial observation: [ 0. 25.  0.]
Observation: [9.         0.         1.32323445], Rewards: [-22.99785586770244, -23.850284613551874, -23.323234452594743], Done: False, Info: {}
Number of people: 9, Current temperature: 0.0, Current load: 1.3232344525947421, Hours: 1
Observation: [8.         1.06687225 1.06687225], Rewards: [-917.410631254857, -918.0172200507184, -919.8662554965649], Done: False, Info: {}
Number of people: 8, Current temperature: 1.0668722517175468, Current load: 1.0668722517175468, Hours: 2
Observation: [1.         0.25256196 0.12628098], Rewards: [-820.1422858434463, -821.8616756753554, -821.6211570655294], Done: False, Info: {}
Number of people: 1, Current temperature: 0.2525619563137824, Current load: 0.1262809781568912, Hours: 3
Observation: [0.         0.15406898 0.05135633], Rewards: [-121.60088586331369, -122.0, -121.79457469580153], Done: False, Info: {}
Number of people: 0, Current temperature: 0.15406897814885312, Current load: 0.05135632604961771, Hours: 4

In [ ]:
import gym
from gym import spaces
import numpy as np

class RoomEnvironment(gym.Env):
    def __init__(self):
        super(RoomEnvironment, self).__init__()

        # Define action and observation space
        self.action_space = spaces.Discrete(4)  # 0: Increase load, 1: Decrease load, 2: Increase temperature, 3: Decrease temperature
        self.observation_space = spaces.Box(low=np.array([0, 15, 0]), high=np.array([10, 30, 100]), dtype=np.float32)

        # Initial state
        self.num_people = 5
        self.temperature = 22.0
        self.electricity_load = 50.0

        # Constants
        self.max_people = 10
        self.min_people = 0
        self.max_temperature = 30.0
        self.min_temperature = 15.0
        self.max_load = 100.0
        self.min_load = 0.0

        # Maximum number of Steps per Episode
        self.max_steps = 100
        self.step_count = 0

    def step(self, action):
        # Apply action
        if action == 0:
            self.electricity_load = min(self.electricity_load + 10, self.max_load)
        elif action == 1:
            self.electricity_load = max(self.electricity_load - 10, self.min_load)
        elif action == 2:
            self.temperature = min(self.temperature + 1, self.max_temperature)
        elif action == 3:
            self.temperature = max(self.temperature - 1, self.min_temperature)

        # Update state based on the number of people
        self.update_state()

        # Calculate reward
        reward = self.calculate_reward()

        # Check if the episode is done
        self.step_count += 1
        done = self.step_count >= self.max_steps

        # Optionally we can pass additional info, we are not using that for now
        info = {}

        return self.observe(), reward, done, info

    def reset(self):
        # Reset the state of the environment to an initial state
        self.num_people = np.random.randint(self.min_people, self.max_people + 1)
        self.temperature = np.random.uniform(self.min_temperature, self.max_temperature)
        self.electricity_load = np.random.uniform(self.min_load, self.max_load)
        self.step_count = 0

        return self.observe()

    def render(self, mode='human'):
        # Render the environment to the screen
        print(f"Number of People: {self.num_people}, Temperature: {self.temperature}, Electricity Load: {self.electricity_load}")

    def observe(self):
        # Return the current state as an observation
        return np.array([self.num_people, self.temperature, self.electricity_load], dtype=np.float32)

    def update_state(self):
        # Update the state based on the number of people
        if self.num_people > 7:
            self.temperature += 0.5
            self.electricity_load += 5
        elif self.num_people < 3:
            self.temperature -= 0.5
            self.electricity_load -= 5

    def calculate_reward(self):
        # Calculate reward based on the current state
        temperature_target = 22.0
        load_target = 50.0

        temperature_reward = -abs(self.temperature - temperature_target)
        load_reward = -abs(self.electricity_load - load_target)

        return temperature_reward + load_reward

# Example usage
env = RoomEnvironment()
num_episodes = 200

for episode in range(num_episodes):
    observation = env.reset()
    done = False
    total_reward = 0

    print(f"Episode {episode + 1}")

    while not done:
        action = env.action_space.sample()  # Take a random action
        observation, reward, done, info = env.step(action)
        env.render()
        total_reward += reward

    print(f"Total reward for episode {episode + 1}: {total_reward}\n")

Streaming output truncated to the last 5000 lines.
Number of People: 10, Temperature: 30.5, Electricity Load: 105.0
Number of People: 10, Temperature: 31.0, Electricity Load: 105.0
Number of People: 10, Temperature: 30.5, Electricity Load: 110.0
Number of People: 10, Temperature: 30.0, Electricity Load: 115.0
Number of People: 10, Temperature: 30.5, Electricity Load: 105.0
Number of People: 10, Temperature: 30.5, Electricity Load: 110.0
Number of People: 10, Temperature: 31.0, Electricity Load: 105.0
Number of People: 10, Temperature: 30.5, Electricity Load: 110.0
Number of People: 10, Temperature: 30.0, Electricity Load: 115.0
Number of People: 10, Temperature: 30.5, Electricity Load: 105.0
Number of People: 10, Temperature: 31.0, Electricity Load: 105.0
Number of People: 10, Temperature: 31.5, Electricity Load: 105.0
Number of People: 10, Temperature: 32.0, Electricity Load: 100.0
Number of People: 10, Temperature: 32.5, Electricity Load: 105.0
Number of People: 10, Temperature: 32.0

In [ ]:
import gym
from gym import spaces
import numpy as np

class RoomEnvironment(gym.Env):
    def __init__(self):
        super(RoomEnvironment, self).__init__()

        # Define action and observation space
        self.action_space = spaces.Discrete(4)  # 0: Increase load, 1: Decrease load, 2: Increase temperature, 3: Decrease temperature
        self.observation_space = spaces.Box(low=np.array([0, 15, 0]), high=np.array([10, 30, 100]), dtype=np.float32)

        # Initial state
        self.num_people = 5
        self.temperature = 22.0
        self.electricity_load = 50.0

        # Constants
        self.max_people = 10
        self.min_people = 0
        self.max_temperature = 30.0
        self.min_temperature = 15.0
        self.max_load = 100.0
        self.min_load = 0.0

        # Maximum number of steps per episode
        self.max_steps = 100
        self.step_count = 0

    def step(self, action):
        # Apply action
        if action == 0:
            self.electricity_load = min(self.electricity_load + 10, self.max_load)
        elif action == 1:
            self.electricity_load = max(self.electricity_load - 10, self.min_load)
        elif action == 2:
            self.temperature = min(self.temperature + 1, self.max_temperature)
        elif action == 3:
            self.temperature = max(self.temperature - 1, self.min_temperature)

        # Update state based on the number of people
        self.update_state()

        # Calculate reward
        reward = self.calculate_reward()

        # Check if the episode is done
        self.step_count += 1
        done = self.step_count >= self.max_steps

        # Optionally we can pass additional info, we are not using that for now
        info = {}

        return self.observe(), reward, done, info

    def reset(self):
        # Reset the state of the environment to an initial state
        self.num_people = np.random.randint(self.min_people, self.max_people + 1)
        self.temperature = np.random.uniform(self.min_temperature, self.max_temperature)
        self.electricity_load = np.random.uniform(self.min_load, self.max_load)
        self.step_count = 0

        return self.observe()

    def render(self, mode='human'):
        # Render the environment to the screen
        print(f"Number of People: {self.num_people}, Temperature: {self.temperature}, Electricity Load: {self.electricity_load}")

    def observe(self):
        # Return the current state as an observation
        return np.array([self.num_people, self.temperature, self.electricity_load], dtype=np.float32)

    def update_state(self):
        # Update the state based on the number of people
        if self.num_people > 7:
            self.temperature += 0.5
            self.electricity_load += 5
        elif self.num_people < 3:
            self.temperature -= 0.5
            self.electricity_load -= 5

    def calculate_reward(self):
        # Calculate reward based on the current state
        temperature_target = 22.0
        load_target = 50.0

        temperature_reward = -abs(self.temperature - temperature_target)
        load_reward = -abs(self.electricity_load - load_target)

        return temperature_reward + load_reward

# Example usage
env = RoomEnvironment()
num_episodes = 200

for episode in range(num_episodes):
    observation = env.reset()
    done = False
    total_reward = 0

    print(f"Episode {episode + 1}")

    while not done:
        action = env.action_space.sample()  # Take a random action
        observation, reward, done, info = env.step(action)
        env.render()
        total_reward += reward

    print(f"Total reward for episode {episode + 1}: {total_reward}\n")

Streaming output truncated to the last 5000 lines.
Number of People: 6, Temperature: 29.0, Electricity Load: 14.576553214431996
Number of People: 6, Temperature: 29.0, Electricity Load: 24.576553214431996
Number of People: 6, Temperature: 29.0, Electricity Load: 14.576553214431996
Number of People: 6, Temperature: 29.0, Electricity Load: 4.576553214431996
Number of People: 6, Temperature: 29.0, Electricity Load: 0.0
Number of People: 6, Temperature: 30.0, Electricity Load: 0.0
Number of People: 6, Temperature: 30.0, Electricity Load: 0.0
Number of People: 6, Temperature: 29.0, Electricity Load: 0.0
Number of People: 6, Temperature: 30.0, Electricity Load: 0.0
Number of People: 6, Temperature: 30.0, Electricity Load: 0.0
Number of People: 6, Temperature: 30.0, Electricity Load: 10.0
Number of People: 6, Temperature: 30.0, Electricity Load: 10.0
Number of People: 6, Temperature: 30.0, Electricity Load: 10.0
Number of People: 6, Temperature: 30.0, Electricity Load: 20.0
Number of People: 

In [17]:
pip install gym numpy tensorflow keras-rl2

In [1]:
pip install keras-rl

DDPG

In [ ]:
!pip install gym numpy tensorflow keras-rl2

In [11]:
import gym
from gym import spaces
import numpy as np

class RoomEnvironment(gym.Env):
    def __init__(self):
        super(RoomEnvironment, self).__init__()

        # Define action and observation space
        self.action_space = spaces.Box(low=np.array([-10, -1]), high=np.array([10, 1]), dtype=np.float32)
        self.observation_space = spaces.Box(low=np.array([0, 15, 0]), high=np.array([10, 30, 100]), dtype=np.float32)

        # Initial state
        self.num_people = 5
        self.temperature = 22.0
        self.electricity_load = 50.0

        # Constants
        self.max_people = 10
        self.min_people = 0
        self.max_temperature = 30.0
        self.min_temperature = 15.0
        self.max_load = 100.0
        self.min_load = 0.0

        # Maximum number of steps per episode
        self.max_steps = 100
        self.step_count = 0

    def step(self, action):
        # Apply action
        action = np.clip(action, a_min=-10, a_max=10)
        self.electricity_load = np.clip(self.electricity_load + action[0], self.min_load, self.max_load)
        self.temperature = np.clip(self.temperature + action[1], self.min_temperature, self.max_temperature)

        # Update state based on the number of people
        self.update_state()

        # Calculate reward
        reward = self.calculate_reward()

        # Check if the episode is done
        self.step_count += 1
        done = self.step_count >= self.max_steps

        # Optionally we can pass additional info, we are not using that for now
        info = {}

        return self.observe(), reward, done, info

    def reset(self):
        # Reset the state of the environment to an initial state
        self.num_people = np.random.randint(self.min_people, self.max_people + 1)
        self.temperature = np.random.uniform(self.min_temperature, self.max_temperature)
        self.electricity_load = np.random.uniform(self.min_load, self.max_load)
        self.step_count = 0

        return self.observe()

    def render(self, mode='human'):
        # Render the environment to the screen
        print(f"Number of People: {self.num_people}, Temperature: {self.temperature}, Electricity Load: {self.electricity_load}")

    def observe(self):
        # Return the current state as an observation
        return np.array([self.num_people, self.temperature, self.electricity_load], dtype=np.float32)

    def update_state(self):
        # Update the state based on the number of people
        if self.num_people > 7:
            self.temperature += 0.5
            self.electricity_load += 5
        elif self.num_people < 3:
            self.temperature -= 0.5
            self.electricity_load -= 5

    def calculate_reward(self):
        # Calculate reward based on the current state
        temperature_target = 22.0
        load_target = 50.0

        temperature_reward = -abs(self.temperature - temperature_target)
        load_reward = -abs(self.electricity_load - load_target)

        return temperature_reward + load_reward

Define the DDPG Agent

In [5]:
!apt-get update && apt-get install ffmpeg freeglut3-dev xvfb  # For visualization
!pip install "stable-baselines3[extra]>=2.0.0a4"

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:3 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packages [2,021 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.3 MB]
Get:10 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1,545 kB]
Get:11 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1,373 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 Pa

In [6]:
import stable_baselines3

print(f"{stable_baselines3.__version__=}")

stable_baselines3.__version__='2.9.0'


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [7]:

import gymnasium as gym
import numpy as np
import torch

from stable_baselines3 import DDPG
from stable_baselines3.common.noise import (
    OrnsteinUhlenbeckActionNoise
)

# --------------------------------------------------
# 1. Initialize a continuous-action environment
# --------------------------------------------------
env = gym.make("Pendulum-v1")

nb_actions = env.action_space.shape[0]

# --------------------------------------------------
# 2. Configure Ornstein-Uhlenbeck exploration noise
# --------------------------------------------------
action_noise = OrnsteinUhlenbeckActionNoise(
    mean=np.zeros(nb_actions),
    sigma=0.3 * np.ones(nb_actions),
    theta=0.15,
    dt=0.01
)

# --------------------------------------------------
# 3. Create the DDPG agent
# --------------------------------------------------
policy_kwargs = dict(
    net_arch=dict(
        pi=[400, 300],
        qf=[400, 300]
    )
)

agent = DDPG(
    policy="MlpPolicy",
    env=env,
    learning_rate=1e-3,
    buffer_size=100_000,
    learning_starts=1_000,
    batch_size=64,
    gamma=0.99,
    tau=0.001,
    action_noise=action_noise,
    policy_kwargs=policy_kwargs,
    verbose=1
)

# --------------------------------------------------
# 4. Train the agent
# --------------------------------------------------
agent.learn(total_timesteps=100_000)

# --------------------------------------------------
# 5. Test the trained agent
# --------------------------------------------------
for episode in range(10):
    obs, info = env.reset()
    total_reward = 0.0

    for step in range(100):
        action, _ = agent.predict(
            obs, deterministic=True
        )

        obs, reward, terminated, truncated, info = (
            env.step(action)
        )

        total_reward += reward

        if terminated or truncated:
            break

    print(
        f"Episode {episode + 1}: "
        f"reward = {total_reward:.2f}"
    )

env.close()

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 200      |
|    ep_rew_mean     | -1.3e+03 |
| time/              |          |
|    episodes        | 4        |
|    fps             | 2656     |
|    time_elapsed    | 0        |
|    total_timesteps | 800      |
---------------------------------


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------------
| rollout/           |           |
|    ep_len_mean     | 200       |
|    ep_rew_mean     | -1.37e+03 |
| time/              |           |
|    episodes        | 8         |
|    fps             | 169       |
|    time_elapsed    | 9         |
|    total_timesteps | 1600      |
| train/             |           |
|    actor_loss      | 9.7       |
|    critic_loss     | 0.0374    |
|    learning_rate   | 0.001     |
|    n_updates       | 599       |
----------------------------------
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 200       |
|    ep_rew_mean     | -1.43e+03 |
| time/              |           |
|    episodes        | 12        |
|    fps             | 107       |
|    time_elapsed    | 22        |
|    total_timesteps | 2400      |
| train/             |           |
|    actor_loss      | 13.1      |
|    critic_loss     | 0.025     |
|    learning_rate   | 0.001     |
|    n_updates      

In [3]:
!pip install pythreejs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 271.7/271.7 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 92.8 MB/s eta 0:00:00


3D rendering environments

In [9]:

from pythreejs import (
    Mesh,
    BoxGeometry,
    MeshBasicMaterial,
    Scene,
    AmbientLight,
    Renderer,
    PerspectiveCamera,
    OrbitControls
)

cube = Mesh(
    BoxGeometry(1, 1, 1),
    MeshBasicMaterial(color="red")
)

scene = Scene(children=[
    cube,
    AmbientLight()
])

camera = PerspectiveCamera(
    position=[3, 3, 3],
    aspect=1
)

controls = OrbitControls(
    controlling=camera
)

renderer = Renderer(
    camera=camera,
    scene=scene,
    controls=[controls],
    width=600,
    height=400
)

renderer

Renderer(camera=PerspectiveCamera(position=(3.0, 3.0, 3.0), projectionMatrix=(1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.…

Smart-room DDPG implementation with interractive 3D room

This model simulates indoor temperature, outdoor temperature, occupancy, HVAC energy, and a controllable smart plug. It trains the agent in simulation before replaying its learned actions in a 3D room.

The environment and reward function are illustrative, not calibrated to a real building. They can later be replaced with your smart-meter measurements.

In [10]:

import time
import numpy as np
import gymnasium as gym
from gymnasium import spaces

from stable_baselines3 import DDPG
from stable_baselines3.common.noise import OrnsteinUhlenbeckActionNoise

from pythreejs import (
    BoxGeometry, Mesh, MeshStandardMaterial,
    Scene, AmbientLight, DirectionalLight,
    PerspectiveCamera, OrbitControls, Renderer
)
from IPython.display import display


# ============================================================
# 1. SMART-ROOM ENVIRONMENT
# ============================================================

class RoomEnvironment(gym.Env):
    """
    Simulated smart room.

    Observation:
        Indoor temperature, outdoor temperature,
        occupancy, hour sine, hour cosine.

    Action:
        action[0]: HVAC heating/cooling command [-1, 1]
        action[1]: smart-plug command [-1, 1]
    """

    metadata = {"render_modes": []}

    def __init__(self):
        super().__init__()

        self.observation_space = spaces.Box(
            low=np.array([10, -10, 0, -1, -1], dtype=np.float32),
            high=np.array([40, 45, 1, 1, 1], dtype=np.float32),
            dtype=np.float32
        )

        self.action_space = spaces.Box(
            low=-1.0,
            high=1.0,
            shape=(2,),
            dtype=np.float32
        )

        self.target_temp = 22.0
        self.dt_hours = 0.25  # 15-minute time step
        self.max_steps = 96   # One simulated day

        self.reset()

    def _observation(self):
        hour = (self.step_count * 0.25) % 24

        return np.array([
            self.indoor_temp,
            self.outdoor_temp,
            self.occupancy,
            np.sin(2 * np.pi * hour / 24),
            np.cos(2 * np.pi * hour / 24)
        ], dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.step_count = 0
        self.indoor_temp = 19.0
        self.outdoor_temp = 12.0
        self.occupancy = 1.0

        self.hvac_command = 0.0
        self.plug_on = 0

        return self._observation(), {}

    def step(self, action):
        hvac = float(np.clip(action[0], -1, 1))
        plug_on = int(action[1] > 0.0)

        # Time of day: outdoor temperature varies over 24 hours.
        hour = (self.step_count * 0.25) % 24

        self.outdoor_temp = (
            12.0 + 6.0 * np.sin(
                2 * np.pi * (hour - 8) / 24
            )
        )

        # Illustrative room thermal model.
        # Positive HVAC heats; negative HVAC cools.
        passive_change = (
            0.035 * (self.outdoor_temp - self.indoor_temp)
        )

        hvac_effect = 0.65 * hvac

        self.indoor_temp += passive_change + hvac_effect
        self.indoor_temp = float(
            np.clip(self.indoor_temp, 10, 40)
        )

        # Illustrative electrical power in kW.
        hvac_power_kw = 1.5 * abs(hvac)
        plug_power_kw = 0.08 * plug_on

        # Energy consumed during this 15-minute step.
        energy_kwh = (
            hvac_power_kw + plug_power_kw
        ) * self.dt_hours

        # Simplified occupancy schedule.
        self.occupancy = float(
            1.0 if 7 <= hour < 23 else 0.0
        )

        # Penalize temperature deviation when occupied.
        comfort_penalty = (
            self.occupancy
            * ((self.indoor_temp - self.target_temp) ** 2)
        )

        # Discourage unnecessary plug use when unoccupied.
        wasted_plug_penalty = (
            plug_on * (1 - self.occupancy)
        )

        # Reward balances comfort and energy use.
        reward = -(
            0.12 * comfort_penalty
            + 4.0 * energy_kwh
            + 0.2 * wasted_plug_penalty
        )

        self.hvac_command = hvac
        self.plug_on = plug_on
        self.step_count += 1

        terminated = False
        truncated = self.step_count >= self.max_steps

        info = {
            "temperature": self.indoor_temp,
            "energy_kwh": energy_kwh,
            "hvac_power_kw": hvac_power_kw,
            "plug_power_kw": plug_power_kw,
            "occupancy": self.occupancy
        }

        return (
            self._observation(),
            float(reward),
            terminated,
            truncated,
            info
        )


# ============================================================
# 2. CREATE AND TRAIN THE DDPG AGENT
# ============================================================

env = RoomEnvironment()

action_noise = OrnsteinUhlenbeckActionNoise(
    mean=np.zeros(env.action_space.shape[0]),
    sigma=0.2 * np.ones(env.action_space.shape[0]),
    theta=0.15,
    dt=0.01
)

agent = DDPG(
    policy="MlpPolicy",
    env=env,
    learning_rate=1e-3,
    buffer_size=100_000,
    learning_starts=1_000,
    batch_size=64,
    gamma=0.99,
    tau=0.001,
    action_noise=action_noise,
    policy_kwargs={
        "net_arch": {"pi": [400, 300], "qf": [400, 300]}
    },
    verbose=1,
    seed=42
)

print("Training the smart-room controller...")
agent.learn(total_timesteps=30_000)

agent.save("smart_room_ddpg")
print("Model saved as smart_room_ddpg.zip")


# ============================================================
# 3. CREATE AN INTERACTIVE 3D ROOM
# ============================================================

def make_box(size, position, color):
    obj = Mesh(
        geometry=BoxGeometry(
            width=size[0],
            height=size[1],
            depth=size[2]
        ),
        material=MeshStandardMaterial(color=color)
    )
    obj.position = position
    return obj


# Room structure
floor = make_box(
    (6, 0.1, 5), (0, 0, 0), "#b8c5c9"
)

back_wall = make_box(
    (6, 3, 0.1), (0, 1.5, -2.5), "#e5e9ec"
)

left_wall = make_box(
    (0.1, 3, 5), (-3, 1.5, 0), "#d6e4e8"
)

right_wall = make_box(
    (0.1, 3, 5), (3, 1.5, 0), "#d6e4e8"
)

# Furniture and room equipment
desk = make_box(
    (1.6, 0.15, 0.8), (1.4, 0.9, -0.5), "#8b6848"
)

desk_leg1 = make_box(
    (0.1, 0.8, 0.1), (0.8, 0.45, -0.8), "#555b60"
)

desk_leg2 = make_box(
    (0.1, 0.8, 0.1), (2.0, 0.45, -0.8), "#555b60"
)

hvac_unit = make_box(
    (0.8, 0.35, 0.25), (-2, 2.5, -2.2), "#64748b"
)

plug_light = make_box(
    (0.25, 0.25, 0.25), (2.1, 0.5, 1.1), "#22c55e"
)

# Temperature indicator: height changes with indoor temperature.
temp_bar = make_box(
    (0.25, 0.1, 0.25), (-2.2, 0.2, 1.8), "#22c55e"
)

scene = Scene(
    children=[
        floor, back_wall, left_wall, right_wall,
        desk, desk_leg1, desk_leg2,
        hvac_unit, plug_light, temp_bar,
        AmbientLight(intensity=0.7),
        DirectionalLight(
            color="white",
            intensity=1.0,
            position=[4, 8, 5]
        )
    ]
)

camera = PerspectiveCamera(
    position=[8, 6, 8],
    aspect=1.5,
    fov=45
)

controls = OrbitControls(controlling=camera)

renderer = Renderer(
    camera=camera,
    scene=scene,
    controls=[controls],
    width=800,
    height=550
)

print("3D smart room:")
display(renderer)


# ============================================================
# 4. RUN THE TRAINED AGENT AND UPDATE THE 3D SCENE
# ============================================================

obs, _ = env.reset(seed=7)

total_energy = 0.0
temperature_history = []
energy_history = []

print("Starting the 3D control simulation...")

for step in range(env.max_steps):

    # Predict the action learned by DDPG.
    action, _ = agent.predict(
        obs, deterministic=True
    )

    obs, reward, terminated, truncated, info = env.step(
        action
    )

    temperature = info["temperature"]
    total_energy += info["energy_kwh"]

    temperature_history.append(temperature)
    energy_history.append(total_energy)

    # Visualize HVAC operation.
    if abs(env.hvac_command) < 0.08:
        hvac_unit.material.color = "#64748b"
    elif env.hvac_command > 0:
        hvac_unit.material.color = "#ef4444"  # Heating
    else:
        hvac_unit.material.color = "#38bdf8"  # Cooling

    # Green = plug on; gray = plug off.
    plug_light.material.color = (
        "#22c55e" if env.plug_on else "#64748b"
    )

    # Update temperature bar height and position.
    bar_height = float(
        np.clip((temperature - 10) / 30, 0.05, 1.5)
    )

    temp_bar.scale = [1, bar_height / 0.1, 1]
    temp_bar.position = [
        -2.2, bar_height / 2, 1.8
    ]

    if step % 4 == 0:
        print(
            f"Step {step:02d} | "
            f"Temperature: {temperature:.1f} °C | "
            f"HVAC: {env.hvac_command:+.2f} | "
            f"Plug: {'ON' if env.plug_on else 'OFF'} | "
            f"Energy: {total_energy:.3f} kWh"
        )

    # Allow time for widget updates in Jupyter.
    time.sleep(0.05)

    if terminated or truncated:
        break

print(f"\nTotal simulated energy: {total_energy:.3f} kWh")
print(
    f"Mean indoor temperature: "
    f"{np.mean(temperature_history):.2f} °C"
)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Training the smart-room controller...
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 96       |
|    ep_rew_mean     | -759     |
| time/              |          |
|    episodes        | 4        |
|    fps             | 1446     |
|    time_elapsed    | 0        |
|    total_timesteps | 384      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 96       |
|    ep_rew_mean     | -635     |
| time/              |          |
|    episodes        | 8        |
|    fps             | 1703     |
|    time_elapsed    | 0        |
|    total_timesteps | 768      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 96       |
|    ep_rew_mean     | -615     |
| time/              |          |
|    episodes     

Renderer(camera=PerspectiveCamera(aspect=1.5, fov=45.0, position=(8.0, 6.0, 8.0), projectionMatrix=(1.0, 0.0, …

Starting the 3D control simulation...
Step 00 | Temperature: 19.2 °C | HVAC: +1.00 | Plug: ON | Energy: 0.395 kWh
Step 04 | Temperature: 20.0 °C | HVAC: +1.00 | Plug: ON | Energy: 1.975 kWh
Step 08 | Temperature: 20.6 °C | HVAC: +1.00 | Plug: ON | Energy: 3.555 kWh
Step 12 | Temperature: 21.1 °C | HVAC: +1.00 | Plug: ON | Energy: 5.135 kWh
Step 16 | Temperature: 21.7 °C | HVAC: +1.00 | Plug: ON | Energy: 6.715 kWh
Step 20 | Temperature: 22.2 °C | HVAC: +1.00 | Plug: ON | Energy: 8.295 kWh
Step 24 | Temperature: 22.9 °C | HVAC: +1.00 | Plug: ON | Energy: 9.875 kWh
Step 28 | Temperature: 23.6 °C | HVAC: +1.00 | Plug: ON | Energy: 11.455 kWh
Step 32 | Temperature: 24.5 °C | HVAC: +1.00 | Plug: ON | Energy: 13.035 kWh
Step 36 | Temperature: 25.4 °C | HVAC: +1.00 | Plug: ON | Energy: 14.615 kWh
Step 40 | Temperature: 26.4 °C | HVAC: +1.00 | Plug: ON | Energy: 16.195 kWh
Step 44 | Temperature: 27.5 °C | HVAC: +1.00 | Plug: ON | Energy: 17.775 kWh
Step 48 | Temperature: 28.5 °C | HVAC: +1.00 

How the learning process works

The agent observes the room state and chooses two continuous actions.

Component:       Role
State:           Indoor/outdoor temperature, occupancy and time of day

Actor:           Selects HVAC and smart-plug commands

Critic:          Estimates the long-term value of state/action combinations

Reward:          Penalizes energy consumption, temperature discomfort and unnecessary plug use

Replay buffer:   Stores past transitions for learning

The initial 30,000 training steps are a starting point, not a guarantee of optimal performance. The model should be evaluated over multiple days and random seeds.

The thermal model is simplified, the occupancy schedule is fixed, and the smart plug represents a small controllable load. The simulation does not yet use real smart-meter readings or establish actual energy savings. Also, pythreejs requires working Jupyter widget support; it may not display correctly in a plain Python terminal.